In [1]:
import os
import numpy as np
import pandas as pd
import keras
from keras import layers

# ==============================================================================
# 1. OFFLINE BATCH TRAIN RECORDS CSV GENERATION ENGINE
# ==============================================================================
csv_file_target = "historical_it_incidents_dataset.csv"

# Force regeneration of a uniform dataset file matrix
if os.path.exists(csv_file_target):
    os.remove(csv_file_target)

print("--- [1/4] Generating Offline CSV Database Training Dataset ---")
dataset_records = []

sentences = [
    "core data database corrupt system panic error code 500",
    "active directory login credentials block access lock denied reset password",
    "switch blown 4th floor router stack dropping connectivity error",
    "workstation screen flashing orange monitor power loss failure",
    "payroll application software module frozen during execution crash logs"
]

# Generate 500 clean uniform row entries instances entirely offline
for i in range(500):
    text_log = sentences[i % 5]
    asset_class = i % 4
    department_org = (i + 1) % 4
    tenure = float(round(np.random.uniform(0.5, 15.0), 1))
    patch_ver = float(round(np.random.uniform(1.0, 22.4), 1))
    
    urgency_class = 0 if "corrupt" in text_log or "blown" in text_log else (1 if "frozen" in text_log else 2)
    routing_target = 2 if asset_class == 2 else (1 if asset_class == 1 else 0)
    base_sla_hours = 2.0 if urgency_class == 0 else (12.0 if urgency_class == 1 else 24.0)
    
    dataset_records.append([text_log, asset_class, department_org, tenure, patch_ver, urgency_class, routing_target, base_sla_hours])
    
df_train = pd.DataFrame(dataset_records, columns=["text_logs", "asset_code", "dept_code", "tenure_years", "patch_version", "target_urgency", "target_routing", "target_sla"])
df_train.to_csv(csv_file_target, index=False)
print(f"[SUCCESS] Clean dataset spreadsheet generated locally: '{csv_file_target}'")

# ==============================================================================
# 2. LOAD DATA CHANNELS FROM OFFLINE SPREADSHEETS
# ==============================================================================
print("\n--- [2/4] Parsing Historical Data Arrays from CSV ---")
df = pd.read_csv(csv_file_target)

# FIXED: Extracted as 1-dimensional array vectors explicitly to prevent structure leakage
train_text_logs = df["text_logs"].values.astype(str)
train_categorical = df[["asset_code", "dept_code"]].values.astype(np.int32)
train_numerical = df[["tenure_years", "patch_version"]].values.astype(np.float32)

y_urgency = df["target_urgency"].values.astype(np.int32).flatten()
y_routing = df["target_routing"].values.astype(np.int32).flatten()
y_sla = df["target_sla"].values.astype(np.float32).flatten()

# Instantiate text token layers
vectorizer = layers.TextVectorization(max_tokens=2000, output_sequence_length=40)
vectorizer.adapt(train_text_logs)

# FIXED: Passing a 1D list of text directly into the vectorizer outputs a uniform matrix shape
processed_text_vectors = vectorizer(list(train_text_logs)).numpy()

# ==============================================================================
# 3. COMPILING GRAPH LAYERS TOPOLOGY
# ==============================================================================
print("--- [3/4] Initializing 3-Input 3-Output Functional Layers ---")
text_input = layers.Input(shape=(40,), dtype="int32", name="text_input_stream")
text_embed = layers.Embedding(input_dim=2000, output_dim=64)(text_input)
text_features = layers.GlobalAveragePooling1D()(text_embed)

cat_input = layers.Input(shape=(2,), dtype="int32", name="categorical_input_stream")
cat_embed = layers.Embedding(input_dim=50, output_dim=16)(cat_input)
cat_features = layers.Flatten()(cat_embed)

num_input = layers.Input(shape=(2,), dtype="float32", name="numerical_indicators_stream")
num_features = layers.Dense(16, activation="relu")(num_input)

merged = layers.concatenate([text_features, cat_features, num_features])
dense = layers.Dense(64, activation="relu")(merged)
dense = layers.Dropout(0.2)(dense)

urgency_out = layers.Dense(4, activation="softmax", name="urgency_output_head")(dense)
routing_out = layers.Dense(3, activation="softmax", name="routing_output_head")(dense)
sla_out = layers.Dense(1, activation="linear", name="sla_output_head")(dense)

it_mimo_model = keras.Model(inputs=[text_input, cat_input, num_input], outputs=[urgency_out, routing_out, sla_out])

it_mimo_model.compile(
    optimizer="adam",
    loss={
        "urgency_output_head": "sparse_categorical_crossentropy",
        "routing_output_head": "sparse_categorical_crossentropy",
        "sla_output_head": "mean_squared_error"
    }
)

# ==============================================================================
# 4. TRAINING & EXPORT LIFECYCLE
# ==============================================================================
print("--- [4/4] Running Deep Learning Fit Loop Engine ---")
# All inputs and targets are perfectly uniform arrays now.
it_mimo_model.fit(
    x=[processed_text_vectors, train_categorical, train_numerical],
    y=[y_urgency, y_routing, y_sla],
    epochs=5,
    batch_size=32,
    verbose=1
)

it_mimo_model.save("it_mimo_model.keras")
print("\n[SUCCESS] New valid model file 'it_mimo_model.keras' written to disk drive.")


--- [1/4] Generating Offline CSV Database Training Dataset ---
[SUCCESS] Clean dataset spreadsheet generated locally: 'historical_it_incidents_dataset.csv'

--- [2/4] Parsing Historical Data Arrays from CSV ---
--- [3/4] Initializing 3-Input 3-Output Functional Layers ---
--- [4/4] Running Deep Learning Fit Loop Engine ---
Epoch 1/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - loss: 207.5349 - routing_output_head_loss: 3.2546 - sla_output_head_loss: 202.4833 - urgency_output_head_loss: 1.6241
Epoch 2/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 118.9858 - routing_output_head_loss: 2.0659 - sla_output_head_loss: 115.2860 - urgency_output_head_loss: 1.6872
Epoch 3/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 114.8791 - routing_output_head_loss: 1.8863 - sla_output_head_loss: 111.1593 - urgency_output_head_loss: 1.7570
Epoch 4/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 107.3476 - routing_output_head_loss: 1.5863 - sla_output_head_loss: 104.0659 - urgency_output_head_loss: 1.7